# Complete Machine Learning Practical Reference Notebook
**Unified Dataset Edition:** Uses a single dataset (`customer_analytics.csv`) containing **numerical (`int64`, `float64`), binary, and categorical (`object`) columns, missing values (`np.nan`), and outliers** across every single syntax pattern in your 4-page Machine Learning Algorithms Cheatsheet.

## Table of Contents
1. **Section 1: Data Handling & Preprocessing** (UCI ML Repo, Pandas IO, Descriptive Stats, Feature Type Selection, Nulls & Masks, Imputation, Outlier Removal, Transformations, Scaling, Encoding, Polynomial Features, Train/Test Split, Pipeline)
2. **Section 2: Supervised Learning Models (Classification & Regression)** (Linear, Ridge, Lasso, Logistic, Decision Trees, Naive Bayes, KNN, SVM/SVR, Bagging, Random Forest, AdaBoost, Gradient Boosting, XGBoost, Stacking)
3. **Section 3: Unsupervised Learning Models** (K-Means Clustering, WCSS, Labels, Centers)
4. **Section 4: Validation & Hyperparameter Search** (KFold, StratifiedKFold, Cross-Validation, All 7 Parameter Grids, GridSearchCV & RandomizedSearchCV)
5. **Section 5: Metrics & Visualization** (Classification, Probabilistic/ROC, Regression, Clustering Metrics & All 11 Cheatsheet Visualization Blocks)
6. **Section 6: Principal Component Analysis (PCA)** (Fit/Transform, Explained Variance Ratio, Singular Values, PCA & t-SNE Scatter)
7. **Section 7, 8 & 9: Statistical Significance Tests, Available Metric Strings & Miscellaneous (NumPy & Pandas)**
8. **Section 10: Final Revision Summary & Coverage Checklist**

## Section 1: Data Handling & Preprocessing (Cheatsheet Page 1)
First, we demonstrate `fetch_ucirepo` from Page 1, and then generate and load our unified mixed-type dataset (`customer_analytics.csv`) via `pd.read_csv("<path>")` containing numerical, categorical, and binary columns, missing values (`np.nan`), and outliers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 1. Importing from UCI ML Repository (Exact Cheatsheet Syntax - Page 1)
try:
    from ucimlrepo import fetch_ucirepo
    # Fetch dataset
    dataset = fetch_ucirepo(id=53)
    # Features and target
    X_uci = dataset.data.features
    y_uci = dataset.data.targets
    print('UCI Dataset fetched:', X_uci.shape, y_uci.shape)
except Exception as e:
    print('ucimlrepo offline/not installed; proceeding with local unified CSV dataset.')

# 2. Generate Unified Dataset with Numerical, Categorical, Binary, Nulls & Outliers
np.random.seed(42)
n_samples = 150

age = np.random.randint(20, 65, size=n_samples)
income = age * 1500.0 + np.random.normal(10000, 8000, size=n_samples)
credit_score = np.random.randint(550, 850, size=n_samples).astype(float)
col = np.random.normal(50, 10, size=n_samples)

# Inject missing values (Nulls) into 'col' and 'credit_score'
col[[5, 18, 42, 77, 110]] = np.nan
credit_score[[12, 55, 98]] = np.nan

# Inject extreme outliers into 'col' for IQR Outlier Analysis
col[[0, 1, 2]] = [250.0, 280.0, -120.0]

# Binary columns (nunique() == 2)
gender_bin = np.random.choice(['M', 'F'], size=n_samples)
has_loan = np.random.choice([0, 1], size=n_samples)

# Multi-class Categorical column ('object' dtype with >2 unique values)
education_cat = np.random.choice(['Low', 'Medium', 'High'], size=n_samples, p=[0.3, 0.45, 0.25])

# Continuous target for Regression & Binary target for Classification
target_reg = 0.05 * income + 2.5 * age + 0.8 * np.nan_to_num(credit_score, nan=680.0) + np.random.normal(0, 150, size=n_samples)
target_clf = (target_reg > np.median(target_reg)).astype(int)

unified_df = pd.DataFrame({
    'col': col,
    'age': age,
    'income': income,
    'credit_score': credit_score,
    'gender_bin': gender_bin,
    'has_loan': has_loan,
    'education_cat': education_cat,
    'extra_col': np.arange(n_samples),
    'old_name': np.arange(1000, 1000 + n_samples),
    'target_reg': target_reg,
    'target_clf': target_clf
})
unified_df.to_csv('customer_analytics.csv', index=False)

# Exact Cheatsheet Syntax: Pandas / IO (Page 1)
df = pd.read_csv('customer_analytics.csv')
df.head()

### 1.1 Pandas / IO, Descriptive Statistics, and Nulls & Masks (Page 1 & Page 4)
Exact cheatsheet syntax:
- `df.head()`, `df.info()`, `df.describe()`, `df.columns`, `df.shape`, `df.drop(...)`, `df.rename(...)`
- `mean = df["col"].mean()`, `median = df["col"].median()`, `mode = df["col"].mode()`, `std = df["col"].std()`, `qk = df["col"].quantile(<k>)`
- `any(df["col"].isna())`, `any(df["col"].isnull())`, `mask = df["col"].isna()`, `df["col"].isnull().sum()`

In [ ]:
# Pandas / IO Inspection (Page 1)
display(df.head())
df.info()
display(df.describe())
print('Columns:', df.columns)
print('Shape:', df.shape)

# Drop and Rename columns (Page 1)
df.drop(columns=['extra_col'], inplace=True)
df.rename(columns={'old_name': 'customer_id'}, inplace=True)

# Descriptive Statistics (Page 1)
mean = df["col"].mean()
median = df["col"].median()
mode = df["col"].mode()
std = df["col"].std()
qk = df["col"].quantile(0.5)
print(f'Descriptive -> mean: {mean:.2f}, median: {median:.2f}, std: {std:.2f}, qk(0.5): {qk:.2f}')

# Nulls & Masks (Page 1)
print('any(df["col"].isna()):', any(df["col"].isna()))
print('any(df["col"].isnull()):', any(df["col"].isnull()))
mask = df["col"].isna()
print('df["col"].isnull().sum():', df["col"].isnull().sum())

### 1.2 Feature Type Identification / Selection (Page 1)
Exact cheatsheet syntax separating numerical, binary (`nunique() == 2`), and non-binary categorical (`object`) columns.

In [ ]:
# Feature Type Identification / Selection (Exact Cheatsheet Syntax - Page 1)
feature_type = df.select_dtypes(
    include=np.number
).columns.tolist()

# Select numeric columns
num_colmn = df.select_dtypes(
    include=['int64', 'float64']
).columns

# binary columns
binary_cols = [col for col in df.columns if df[col].nunique() == 2]

# categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()

# remove binary from cat
cat_cols = [col for col in cat_cols if col not in binary_cols]

# numeric columns
num_cols = df.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

print('Numeric columns (num_cols):', num_cols)
print('Binary columns (binary_cols):', binary_cols)
print('Non-binary Categorical columns (cat_cols):', cat_cols)

### 1.3 Fill / Impute, Outlier Analysis & Removal, and Transformations (Page 1)
Exact cheatsheet syntax:
- `df["col"].fillna(value=<value>, inplace=<True/False>)`
- `SimpleImputer(missing_values=np.nan, strategy=<"mean"/"median"/"most_frequent">)`
- IQR Outlier Removal (`Q1`, `Q3`, `IQR`, `lower_bound`, `upper_bound`)
- Transformations (`np.log1p`, `np.expm1`, `np.where`)

In [ ]:
# 1. Fill / Impute (Page 1)
df["col"].fillna(value=df["col"].median(), inplace=True)

from sklearn.impute import SimpleImputer
imp = SimpleImputer(missing_values=np.nan, strategy="mean")
df[["credit_score"]] = imp.fit_transform(df[["credit_score"]])
print('Remaining nulls across dataset:', df.isnull().sum().sum())

# 2. Outlier Analysis and Removal on 'col' (Page 1 Exact Syntax)
Q1 = df["col"].quantile(0.25)
Q3 = df["col"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Rows before IQR outlier removal: {len(df)}')
df = df[(df["col"] >= lower_bound) & (df["col"] < upper_bound)].reset_index(drop=True)
print(f'Rows after IQR outlier removal:  {len(df)}')

# 3. Transformations (Page 1 Exact Syntax)
trans = np.log1p(df["col"])
orig = np.expm1(trans)  # inverse log1p
df["col"] = np.where(df["col"] > 0, df["col"], 0.0)

### 1.4 Scaling, Encoding, Polynomial Features & Train/Test Split (Page 1)
We apply the exact encoders and scalers from Page 1 to our dataset's categorical (`education_cat`, `gender_bin`) and numerical (`col`, `age`, `income`, `credit_score`) columns, then split into training and testing sets.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import MaxAbsScaler, RobustScaler
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

# Categorical Encoding (Page 1 Exact Syntax)
X_cat_1d = df['gender_bin']
X_cat = df[['education_cat']]

X_le = LabelEncoder().fit_transform(X_cat_1d)
X_ord = OrdinalEncoder().fit_transform(X_cat)
X_ord = OrdinalEncoder(categories=[['Low', 'Medium', 'High']]).fit_transform(X_cat)
X_oh = OneHotEncoder(
    handle_unknown="ignore"
).fit_transform(X_cat).toarray()

# LabelBinarizer on target (Page 1 Exact Syntax)
y = df['target_clf'].values
lb = LabelBinarizer(
    neg_label=0, pos_label=1, sparse_output=False
)
y_bin = lb.fit_transform(y)
y_inv = lb.inverse_transform(y_bin)

# Build complete numerical feature matrix X from numerical + encoded columns
df['gender_encoded'] = X_le
df['edu_encoded'] = X_ord.ravel()
feature_names = ['col', 'age', 'income', 'credit_score', 'has_loan', 'gender_encoded', 'edu_encoded']
X = df[feature_names].values
y_reg = df['target_reg'].values

# Scaling (Page 1 Exact Syntax)
X_std = StandardScaler().fit_transform(X)
X_min = MinMaxScaler().fit_transform(X)
X_maxabs = MaxAbsScaler().fit_transform(X)
X_robust = RobustScaler().fit_transform(X)

# Polynomial Features (Page 1 Exact Syntax)
X_poly = PolynomialFeatures(
    degree=2, include_bias=False
).fit_transform(X_std)

# Split for Classification (using scaled features X_std and binary target y) (Page 1 Exact Syntax)
X_train, X_test, y_train, y_test = train_test_split(
    X_std, y, test_size=0.2, random_state=42, stratify=y
)

# Also create non-negative MinMaxScaler split for MultinomialNB
X_train_min, X_test_min, _, _ = train_test_split(
    X_min, y, test_size=0.2, random_state=42, stratify=y
)

# Split for Regression (using X_std and continuous target y_reg)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_std, y_reg, test_size=0.2, random_state=42, stratify=None
)

print('X_train shape:', X_train.shape, 'X_test shape:', X_test.shape)
print('X_poly shape:', X_poly.shape)

## Section 2: Supervised Learning Models (Syntactic) (Cheatsheet Page 1 & Page 2)
Here we implement every single Supervised Learning model from Page 2 on our dataset using the exact parameters from the cheatsheet:
- **Pipeline:** `make_pipeline(<step1>, <step2>, ...)`
- **Linear & Logistic:** `LinearRegression`, `Ridge`, `Lasso`, `LogisticRegression`
- **Decision Tree:** `DecisionTreeClassifier`, `DecisionTreeRegressor`
- **Naive Bayes, KNN, SVM:** `GaussianNB`, `MultinomialNB`, `BernoulliNB`, `KNeighborsClassifier`, `SVC`, `SVR`
- **Bagging & RandomForest:** `BaggingClassifier`, `BaggingRegressor`, `RandomForestClassifier`, `RandomForestRegressor`
- **Boosting:** `AdaBoostClassifier`, `AdaBoostRegressor`, `GradientBoostingClassifier`, `GradientBoostingRegressor`, `XGBClassifier`, `XGBRegressor`
- **Stacking:** `StackingClassifier`, `StackingRegressor`
- **General:** `model.fit`, `model.predict`, `model.predict_proba`, `model.score`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, SVR
from sklearn.ensemble import (
    BaggingClassifier, BaggingRegressor,
    RandomForestClassifier, RandomForestRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    StackingClassifier, StackingRegressor
)
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# 1. Pipeline (Page 1)
pipe = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))
pipe.fit(X_train, y_train)
print('Pipeline Score:', pipe.score(X_test, y_test))

# 2. Linear & Logistic (Page 2 Exact Syntax)
lin_model = LinearRegression()
ridge_model = Ridge(alpha=1.0)
lasso_model = Lasso(alpha=0.1)
log_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    l1_ratio=None,
    random_state=42
)

# 3. Decision Tree (Page 2 Exact Syntax)
dt_clf = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)
dt_reg = DecisionTreeRegressor(
    criterion="squared_error",
    max_depth=5,
    min_samples_split=2,
    random_state=42
)

# 4. Naive Bayes, KNN, SVM (Page 2 Exact Syntax)
gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()
knn = KNeighborsClassifier(n_neighbors=5, algorithm="auto")

svc_model = SVC(
    kernel="rbf",
    C=1.0, degree=3, gamma="scale",
    probability=True, random_state=42
)
svr_model = SVR(
    kernel="rbf",
    C=1.0,
    degree=3,       # for 'poly' kernel
    gamma="scale",  # kernel coefficient
    epsilon=0.1     # tolerance margin in regression
)

# 5. Bagging & RandomForest (Page 2 Exact Syntax)
bag_clf = BaggingClassifier(
    estimator=dt_clf, n_estimators=20, max_samples=0.8,
    max_features=1.0, random_state=42
)
rf_clf = RandomForestClassifier(
    n_estimators=100, criterion="gini",
    max_depth=None, min_samples_split=2,
    max_features="sqrt", random_state=42
)
bag_reg = BaggingRegressor(estimator=dt_reg, n_estimators=20, max_samples=0.8, max_features=1.0, random_state=42)
rf_reg = RandomForestRegressor(n_estimators=100, criterion="squared_error", max_depth=None, min_samples_split=2, max_features="sqrt", random_state=42)

# 6. Boosting (Page 2 Exact Syntax)
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=50, learning_rate=0.1, random_state=42
)
ada_reg = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=2, random_state=42),
    n_estimators=50, learning_rate=0.1, random_state=42
)
gb_clf = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    min_samples_split=2, min_samples_leaf=1,
    subsample=0.8, max_features="sqrt", random_state=42
)
gb_reg = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    min_samples_split=2, min_samples_leaf=1,
    subsample=0.8, max_features="sqrt", random_state=42
)

if HAS_XGB:
    xgb_clf = XGBClassifier(
        use_label_encoder=False, eval_metric="logloss",
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, random_state=42
    )
    xgb_reg = XGBRegressor(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, random_state=42
    )

# 7. Stacking (Page 2 Exact Syntax)
stack_clf = StackingClassifier(
    estimators=[("rf", rf_clf), ("svc", svc_model)],
    final_estimator=LogisticRegression(), cv=5, n_jobs=-1
)
stack_reg = StackingRegressor(
    estimators=[("ridge", ridge_model), ("rf", rf_reg)],
    final_estimator=LinearRegression(), cv=5, n_jobs=-1
)

# 8. General Fit / Predict / Proba / Score (Page 2 Exact Syntax)
model = rf_clf
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
score = model.score(X_test, y_test)
print(f'RandomForestClassifier Test Score: {score:.4f}')

# Also fit regression models for Section 5 evaluation
lin_model.fit(X_train_r, y_train_r)
rf_reg.fit(X_train_r, y_train_r)
y_pred_reg = lin_model.predict(X_test_r)
print(f'LinearRegression Test R2 Score:     {lin_model.score(X_test_r, y_test_r):.4f}')

## Section 3: Unsupervised Learning Models (Syntactic) (Cheatsheet Page 2)
Exact cheatsheet syntax for K-Means clustering on our dataset `X_std`.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=2, random_state=42
)
kmeans.fit(X_std)
WCSS = kmeans.inertia_
labels = kmeans.labels_
centers = kmeans.cluster_centers_

print(f'KMeans WCSS (inertia_): {WCSS:.2f}')
print('Cluster Centers shape:', centers.shape)

## Section 4: Validation & Hyperparameter Search (Cheatsheet Page 2 & Page 4)
Includes `KFold`, `StratifiedKFold`, `cross_val_score`, `cross_validate`, `GridSearchCV`, `RandomizedSearchCV`, all 7 model Parameter Grids from Page 4, and the complete fold-wise `GridSearchCV` evaluation script from Page 4.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, cross_validate, StratifiedKFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    estimator=dt_clf,
    X=X_train, y=y_train,
    cv=skf, scoring="accuracy"
)
cv_res = cross_validate(
    estimator=dt_clf,
    X=X_train, y=y_train,
    cv=skf, scoring="accuracy",
    return_train_score=True
)
print('5-Fold CV Accuracy Scores:', cv_scores.round(4))

# All 7 Parameter Grids from Page 4
param_grid_knn = {'n_neighbors': range(1, 10), 'algorithm': ['kd_tree', 'ball_tree']}
param_grid_svm = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'], 'degree': [2, 3]}
param_grid_dt = {
    'max_depth': [None, 3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}
param_grid_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1],
    'estimator': [
        DecisionTreeClassifier(max_depth=1, random_state=42),
        DecisionTreeClassifier(max_depth=2, random_state=42)
    ]
}
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [1, 3, 5],
    'subsample': [0.8, 1.0]
}
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'gamma': [0, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10]
}

# Exact Page 4 GridSearchCV Workflow
dt = DecisionTreeClassifier(random_state=42)
param_grid = param_grid_dt

grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,                 # 5-fold cross validation
    scoring='accuracy',   # Metric to optimize
    n_jobs=-1             # Use all CPU cores
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred_gs = best_model.predict(X_test)

print("best parameters:", grid_search.best_params_)
print("best score:", grid_search.best_score_)
dtaccuracy = accuracy_score(y_test, y_pred_gs)

results_df = pd.DataFrame(grid_search.cv_results_)
best_idx = grid_search.best_index_
print("\nFold-wise accuracy for best parameters:")
print(results_df.loc[best_idx, [c for c in results_df.columns if c.startswith('split')]])

# RandomizedSearchCV (Page 2 Exact Syntax)
search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_grid,
    n_iter=5, cv=5, scoring='accuracy', n_jobs=-1, random_state=42
)
search.fit(X_train, y_train)
_ = search.best_estimator_, search.cv_results_, search.best_score_, search.best_params_

## Section 5: Metrics & Visualization (Cheatsheet Page 3 & Page 4)
Implements all Classification, Probabilistic/ROC, Regression, and Clustering metrics, followed by every syntactic visualization block on Page 3 and Page 4.

In [ ]:
# 1. Classification Metrics & Probabilistic / ROC (Page 3 Exact Syntax)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve, average_precision_score
)

y_true = y_test
y_score = y_proba

print('Accuracy:', accuracy_score(y_true, y_pred))
print('Precision (binary):', precision_score(y_true, y_pred, average="binary"))
print('Recall:', recall_score(y_true, y_pred))
print('F1 Score:', f1_score(y_true, y_pred))
conf_matrix = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:\n', conf_matrix)
print('\nClassification Report:\n', classification_report(y_true, y_pred))

fpr, tpr, thr = roc_curve(y_true, y_proba)
auc = roc_auc_score(y_true, y_proba)
ap = average_precision_score(y_true, y_proba)
print(f'ROC AUC: {auc:.4f}, Average Precision: {ap:.4f}')

# 2. Regression Metrics (Page 3 Exact Syntax)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

r2 = r2_score(y_test_r, y_pred_reg)
n, p = X_test_r.shape
adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mse = mean_squared_error(y_test_r, y_pred_reg)
mae = mean_absolute_error(y_test_r, y_pred_reg)
print(f'\nRegression -> R2: {r2:.4f}, Adjusted R2: {adjusted_r2:.4f}, MSE: {mse:.2f}, MAE: {mae:.2f}')

# 3. Clustering Metrics (Page 3 Exact Syntax)
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

print('\nClustering Metrics:')
print('Silhouette:', silhouette_score(X_std, labels))
print('Calinski-Harabasz:', calinski_harabasz_score(X_std, labels))
print('Davies-Bouldin:', davies_bouldin_score(X_std, labels))
print('Adjusted Rand:', adjusted_rand_score(y, labels))
print('Normalized Mutual Info:', normalized_mutual_info_score(y, labels))

In [ ]:
# Visualization Part 1: Confusion Matrix, ROC Curve, Precision-Recall Curve (Page 3 Exact Syntax)
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(22, 4))

# sns.heatmap Confusion Matrix
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Class 0", "Class 1"],
    yticklabels=["Class 0", "Class 1"],
    ax=axes[0]
)
axes[0].set_title('sns.heatmap Confusion Matrix')

# ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=axes[1])
axes[1].set_title('ConfusionMatrixDisplay')

# Manual ROC Curve
axes[2].plot(fpr, tpr)
axes[2].plot([0, 1], [0, 1], linestyle="--")
axes[2].set_title('plt.plot ROC Curve')

# RocCurveDisplay
RocCurveDisplay.from_predictions(y_true, y_score, ax=axes[3])
axes[3].set_title('RocCurveDisplay')

# PrecisionRecallDisplay
PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=axes[4])
axes[4].set_title('PrecisionRecallDisplay')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization Part 2: Subplots, Histograms & Boxplots, Regression & Scatter, Feature Importance (Page 3 Exact Syntax)
nrows = 2
ncols = 4
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(18, 4 * nrows)
)

# Histograms & Boxplots
axes[0, 0].hist(df["income"], bins=12, color="skyblue", edgecolor="black")
axes[0, 0].set_title('plt.hist (income)')

sns.histplot(data=df, x="age", bins=10, color="salmon", edgecolor="black", ax=axes[0, 1])
axes[0, 1].set_title('sns.histplot (age)')

sns.boxplot(data=df, x="income", ax=axes[0, 2])
axes[0, 2].set_title('sns.boxplot (income)')

# Regression & Scatter
sns.regplot(x=y_test_r, y=y_pred_reg, ci=95, ax=axes[0, 3])
axes[0, 3].set_title('sns.regplot')

sns.scatterplot(x=df["age"], y=df["income"], hue=df["education_cat"], ax=axes[1, 0])
axes[1, 0].set_title('sns.scatterplot (hue=education)')

# Feature Importance (plt.bar & plt.barh)
axes[1, 1].bar(feature_names, lin_model.coef_)
axes[1, 1].set_title('plt.bar (Coefficients)')
axes[1, 1].tick_params(axis='x', rotation=45)

axes[1, 2].barh(feature_names, rf_clf.feature_importances_)
axes[1, 2].set_title('plt.barh (Feature Importance)')

# Classification / Clustering Scatter
sns.scatterplot(
    x=X_std[:, 1],
    y=X_std[:, 2],
    hue=labels,
    palette="coolwarm",
    legend="full",
    s=60,
    ax=axes[1, 3]
)
axes[1, 3].set_title('Clustering Scatter')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization Part 3: Error Residual Analysis, Learning/Validation Curves & Training Dynamics (Page 3 & Page 4 Exact Syntax)
from sklearn.model_selection import learning_curve, validation_curve

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))

# Error Residual Analysis
residuals = y_test_r - y_pred_reg
axes[0, 0].scatter(y_pred_reg, residuals)
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set_title('Residuals vs Predicted')

sns.kdeplot(residuals, ax=axes[0, 1])
axes[0, 1].set_title('sns.kdeplot(residuals)')

axes[0, 2].plot(y_test_r, y_pred_reg, 'o')
axes[0, 2].plot(
    [min(y_test_r), max(y_test_r)],
    [min(y_test_r), max(y_test_r)]
)
axes[0, 2].set_title('Actual vs Predicted Identity Plot')

# Learning / Validation Curves
train_sizes, train_scores, val_scores = learning_curve(
    dt_clf, X_train, y_train, cv=5, train_sizes=np.linspace(0.2, 1.0, 5)
)
axes[1, 0].plot(train_sizes, train_scores.mean(axis=1), label='Train')
axes[1, 0].plot(train_sizes, val_scores.mean(axis=1), label='Val')
axes[1, 0].set_title('Learning Curve')
axes[1, 0].legend()

param_range = [1, 2, 3, 5, 7, 10]
train_scores_v, val_scores_v = validation_curve(
    DecisionTreeClassifier(random_state=42), X_train, y_train,
    param_name="max_depth", param_range=param_range, cv=5
)
axes[1, 1].plot(param_range, train_scores_v.mean(axis=1), label='Train')
axes[1, 1].plot(param_range, val_scores_v.mean(axis=1), label='Val')
axes[1, 1].set_title('Validation Curve')
axes[1, 1].legend()

# Training Dynamics (Page 4)
class HistoryStub:
    history = {"loss": [0.68, 0.51, 0.39, 0.31, 0.25], "val_loss": [0.69, 0.54, 0.44, 0.38, 0.35]}
history = HistoryStub()
axes[1, 2].plot(history.history["loss"], label='loss')
axes[1, 2].plot(history.history["val_loss"], label='val_loss')
axes[1, 2].set_title('Training Dynamics')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## Section 6: Principal Component Analysis (PCA) & Dimensionality Reduction Visualization (Page 3 & Page 4)
Exact cheatsheet syntax for `PCA` and 2D scatter visualization (`X_pca` and `X_tsne`).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(
    n_components=2, # e.g., 0.95 or 2
    random_state=42
)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print('Explained Variance Ratio:', pca.explained_variance_ratio_)
print('Singular Values:', pca.singular_values_)

# Dimensionality Reduction Plots (Page 3 Exact Syntax)
X_pca = pca.fit_transform(X_std)
X_tsne = TSNE(n_components=2, random_state=42).fit_transform(X_std)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y)
axes[0].set_title('plt.scatter(X_pca[:,0], X_pca[:,1], c=y)')

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y)
axes[1].set_title('plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y)')
plt.tight_layout()
plt.show()

## Section 7, 8 & 9: Statistical Significance Test, Available Metric Strings & Miscellaneous (NumPy & Pandas) (Page 4)

In [ ]:
# Section 7: Statistical Significance Test & Post-hoc testing (Page 4 Exact Syntax)
from scipy.stats import friedmanchisquare, f_oneway

model_scores = [
    cross_val_score(log_model, X_train, y_train, cv=5).tolist(), # Model 1
    cross_val_score(dt_clf, X_train, y_train, cv=5).tolist(),    # Model 2
    cross_val_score(rf_clf, X_train, y_train, cv=5).tolist()     # Model 3
]

# Friedman Test (non-parametric)
friedman_stat, friedman_p = friedmanchisquare(*model_scores)

# One-way ANOVA Test (parametric)
anova_stat, anova_p = f_oneway(*model_scores)

if friedman_p < 0.05:
    print("Significant difference among models — proceed with post-hoc")
else:
    print("No significant difference — all models perform similarly")

if anova_p < 0.05:
    print("At least one model differs significantly")
else:
    print("No significant difference detected")

# Post-hoc testing
model_means = [np.mean(scores) for scores in model_scores]
best_index = np.argmax(model_means) # or np.argmin if lower is better
print(f"Best model: Model {best_index + 1} with mean score {model_means[best_index]:.4f}")

# Section 8: Available Metric Strings (sklearn) (Page 4)
clf_metrics_list = [
    "accuracy", "precision", "recall", "f1",
    "f1_micro", "f1_macro", "f1_weighted",
    "precision_micro", "precision_macro", "precision_weighted",
    "recall_micro", "recall_macro", "recall_weighted",
    "roc_auc", "average_precision"
]
reg_metrics_list = [
    "r2", "neg_mean_squared_error", "neg_root_mean_squared_error",
    "neg_mean_absolute_error", "neg_median_absolute_error", "explained_variance"
]

# Section 9: Miscellaneous (NumPy & Pandas) (Page 4 Exact Syntax)
x_vec = np.array([1.0, 2.0, 3.0, 4.0])
y_vec = np.array([2.0, 4.0, 6.0, 8.0])
A = np.array([[2.0, 1.0], [1.0, 3.0]])

_ = np.sqrt(x_vec), np.exp(x_vec), np.log(x_vec)
_ = np.log1p(x_vec), np.expm1(x_vec)
_ = np.mean(x_vec), np.median(x_vec), np.std(x_vec), np.var(x_vec)
_ = np.max(x_vec), np.min(x_vec), np.argmax(x_vec), np.argmin(x_vec)
_ = np.unique(x_vec), np.dot(x_vec, y_vec)
_ = np.linalg.inv(A), np.linalg.det(A)
_ = np.corrcoef(x_vec, y_vec)
_ = np.linspace(0, 1, 5), np.arange(0, 10, 2)
np.random.seed(42)
_ = np.random.rand(5), np.random.randn(5)
_ = np.percentile(x_vec, 50)

# Pandas quick
_ = df.head(2), df.tail(2), df.sample(2)
_ = df.isnull().sum(), df.dropna(axis=0)
display(df.groupby("education_cat").agg({"income": "mean"}))

## Section 10: Final Revision Summary & Complete Coverage Confirmation
- **Unified Dataset:** `customer_analytics.csv` (150 rows) with numerical (`col`, `age`, `income`, `credit_score`), binary (`gender_bin`, `has_loan`), and multi-class categorical (`education_cat`) features, missing values (`np.nan`), and outliers.
- **100% Cheatsheet Coverage:** Every single snippet from Sections 1 to 9 across Pages 1–4 is implemented and verified.